## ERA5 Single Levels of Interest

In [ ]:
import xarray as xr

ds = xr.open_dataset(
    "https://data.earthdatahub.destine.eu/era5/reanalysis-era5-single-levels-v0.zarr",
    storage_options={"client_kwargs":{"trust_env":True}},
    chunks={},
    engine="zarr",
)

ds = ds.sel(latitude=slice(52, 40), longitude=slice(2, 20))
selected_vars = ["t2m", "tp", "ssrd"]  # Modify names as needed
ds_filtered = ds[selected_vars]
ds_filtered = ds_filtered.sel(valid_time=slice("2000-01-01", "2020-12-31"))

ds_daily = xr.Dataset()
ds_daily["t2m"] = ds_filtered["t2m"].resample(valid_time="1D").mean()  # Daily mean
ds_daily["tp"] = ds_filtered["tp"].resample(valid_time="1D").sum()  # Daily sum
ds_daily["ssrd"] = ds_filtered["ssrd"].resample(valid_time="1D").sum()  # Daily sum
ds_daily = ds_daily.drop_vars(["depthBelowLandLayer", "entireAtmosphere", "number", "surface"])
ds_daily.attrs["crs"] = "EPSG:4326"
ds_daily = ds_daily.rename({"latitude": "lat", "longitude": "lon", "valid_time": "time"})
# Define desired chunking (adjust dimensions as needed)
chunk_dict = {"time": 500, "lat": 49, "lon": 73}  # example chunk sizes

# Rechunk uniformly
ds_daily_chunked = ds_daily.chunk(chunk_dict)

## PUSHED TO RASTER2STAC to the InterTwin STAC

stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

cube = local_conn.load_stac(
    url=stac_item,
    bands=["data"]
).execute()
cube


## ERA5 Pressure Levels of Interest

In [ ]:
import xarray as xr

ds = xr.open_dataset(
    "https://data.earthdatahub.destine.eu/era5/reanalysis-era5-pressure-levels-v0.zarr",
    storage_options={"client_kwargs":{"trust_env":True}},
    chunks={},
    engine="zarr",
)

ds = ds.sel(latitude=slice(52, 40), longitude=slice(2, 20))
# Select required isobaric levels
selected_isobars = [850]
ds_filtered = ds.sel(isobaricInhPa=selected_isobars)  # Select only the required levels
ds_filtered = ds_filtered.sel(valid_time=slice("2000-01-01", "2020-12-31"))
selected_vars = ["q", "t", "u", "v", "z"]  # Modify names as needed
ds_filtered = ds_filtered[selected_vars]
ds_daily_pressure = ds_filtered.resample(valid_time='1D').mean()
ds_daily_pressure = ds_daily_pressure.squeeze()
ds_daily_pressure = ds_daily_pressure.drop_vars(["isobaricInhPa", "number"])
ds_daily_pressure = ds_daily_pressure.rename({"latitude": "lat", "longitude": "lon", "valid_time": "time"})
ds_daily_pressure = ds_daily_pressure.rename({var: f"{var}_850" for var in ds_daily_pressure.data_vars})

chunk_dict = {"time": 500, "lat": 49, "lon": 73}  # example chunk sizes

# Rechunk uniformly
ds_daily_pressure = ds_daily_pressure.chunk(chunk_dict)
ds_daily_pressure.attrs["crs"] = "EPSG:4326"
ds_daily_pressure

## PUSHED TO RASTER2STAC to the InterTwin STAC

stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

cube = local_conn.load_stac(
    url=stac_item,
    bands=["data"]
).execute()
cube